In [1]:
# ===========================
# Step 1: Import Libraries
# ===========================

import pandas as pd

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

import json

# JSON file banane ke liye json library use hoti hai.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# ===========================
# Step 2: Load CSV Files
# ===========================

source_df = pd.read_csv("medical_dataset_source_schema.csv")
target_df = pd.read_csv("medical_dataset_target_schema.csv")

In [3]:
print("Source Dataset")
display(source_df.head())

print("Target Dataset")
display(target_df.head())

Source Dataset


,Record_ID,Patient Age,Sex,Blood Sugar,BP,Body Mass Index,Serum Insulin,Cholesterol Level,Smoking Status,Disease
0,10001,61,M,76,107,24.6,129,155,Smoker,Negative
1,10002,64,M,178,62,18.7,126,179,Smoker,Positive
2,10003,56,M,209,86,23.3,157,121,Smoker,Positive
3,10004,65,F,157,77,21.7,187,146,Smoker,Positive
4,10005,45,M,161,82,32.5,37,237,Smoker,Positive


Target Dataset


,Patient_ID,Age,Gender,Glucose,BloodPressure,BMI,Insulin,Cholesterol,Smoker,Diagnosis
0,1,61,Male,76,107,24.6,129,155,Yes,No Diabetes
1,2,64,Male,178,62,18.7,126,179,Yes,Diabetes
2,3,56,Male,209,86,23.3,157,121,Yes,Diabetes
3,4,65,Female,157,77,21.7,187,146,Yes,Diabetes
4,5,45,Male,161,82,32.5,37,237,Yes,Diabetes


In [4]:
# Shape
print("Source Data Shape: ",source_df.shape)
print("Target Data Shape: ",target_df.shape)

Source Data Shape:  (2000, 10)
Target Data Shape:  (2000, 10)


In [5]:
# Column Names
print(source_df.columns.tolist())
print(target_df.columns.tolist())

['Record_ID', 'Patient Age', 'Sex', 'Blood Sugar', 'BP', 'Body Mass Index', 'Serum Insulin', 'Cholesterol Level', 'Smoking Status', 'Disease']
['Patient_ID', 'Age', 'Gender', 'Glucose', 'BloodPressure', 'BMI', 'Insulin', 'Cholesterol', 'Smoker', 'Diagnosis']


# Data Standardization

In [6]:
ALIAS_DICT = {

    "mr no": "medical record number",
    "mr#": "medical record number",
    "mr number": "medical record number",

    "pt": "patient",
    "pt age": "patient age",
    "pt gender": "patient gender",

    "o2 sat": "oxygen saturation",
    "spo2": "oxygen saturation",

    "hr": "heart rate",

    "bp": "blood pressure",

    "hba1c": "hemoglobin a1c",

    "doc": "doctor",
    "physician": "doctor",

    "cell no": "phone number",
    "mobile no": "phone number",

    "blood sugar": "glucose",

    "sex": "gender",

    "patient id": "record id"

}

In [7]:
import re

# ==========================================================
# Function: Standardize Column Names
# ==========================================================

def standardize_columns(df):

    df = df.copy()

    df.columns = (
        df.columns
        .str.lower()
        .str.replace("_", " ", regex=False)
        .str.replace("-", " ", regex=False)
        .str.replace(r"[^a-z0-9 ]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Apply Alias Dictionary
    df.columns = [ALIAS_DICT.get(col, col) for col in df.columns]

    return df


# ==========================================================
# Apply Standardization
# ==========================================================

source_df = standardize_columns(source_df)

target_df = standardize_columns(target_df)


# ==========================================================
# RESULT CHECK
# ==========================================================

print("Source Columns:")
print(source_df.columns.tolist())

print("\nTarget Columns:")
print(target_df.columns.tolist())

Source Columns:
['record id', 'patient age', 'gender', 'glucose', 'blood pressure', 'body mass index', 'serum insulin', 'cholesterol level', 'smoking status', 'disease']

Target Columns:
['record id', 'age', 'gender', 'glucose', 'bloodpressure', 'bmi', 'insulin', 'cholesterol', 'smoker', 'diagnosis']


# Rapid Fuzz

In [8]:
# pip install rapidfuzz

from rapidfuzz import fuzz

# ==========================================================
# STEP : RapidFuzz (Spelling Similarity)
# ==========================================================

# NOTE:
# source_df aur target_df pehle hi standardize_columns()
# function se clean ho chuke hain.
# Isliye RapidFuzz cleaned column names compare karega.

source_columns = source_df.columns.tolist()
target_columns = target_df.columns.tolist()

print("RapidFuzz Similarity Scores\n")

for source_col in source_columns:

    print("-" * 60)
    print("Source Column:", source_col)
    print("-" * 60)

    for target_col in target_columns:

        score = fuzz.ratio(source_col, target_col)

        print(f"{target_col:30} --> {score:.2f}")

    print()

RapidFuzz Similarity Scores

------------------------------------------------------------
Source Column: record id
------------------------------------------------------------
record id                      --> 100.00
age                            --> 16.67
gender                         --> 26.67
glucose                        --> 25.00
bloodpressure                  --> 27.27
bmi                            --> 16.67
insulin                        --> 12.50
cholesterol                    --> 30.00
smoker                         --> 26.67
diagnosis                      --> 22.22

------------------------------------------------------------
Source Column: patient age
------------------------------------------------------------
record id                      --> 20.00
age                            --> 42.86
gender                         --> 35.29
glucose                        --> 22.22
bloodpressure                  --> 25.00
bmi                            --> 14.29
insulin          

In [9]:
# ==========================================================
# NOTE:
# RapidFuzz sirf spelling (text) similarity check karta hai.
# Ye words ka meaning (semantic meaning) nahi samajhta.
#
# Example:
# "patient age" aur "patient id"
# Dono me "patient" common hai, isliye score high aata hai.
# Lekin meaning different hai.
#
# Isi tarah:
# "sex" aur "smoker"
# Kuch letters similar hone ki wajah se score aa sakta hai,
# lekin dono ka matlab bilkul alag hai.
#
# RapidFuzz ki limitations:
# ------------------------------------------
# ❌ patient age  -> patient id   (Wrong Match)
# ❌ sex          -> smoker       (Wrong Match)
# ❌ blood sugar  -> bloodpressure (Wrong Match)
# ❌ body mass index -> bloodpressure (Wrong Match)
#
# RapidFuzz semantic meaning nahi samajhta.
# Ye sirf characters aur spelling compare karta hai.
#
# Isliye RapidFuzz ko final decision ke liye use nahi karenge.
# Ye sirf ek initial (supporting) similarity score dega.
#
# Next step me Sentence Transformer use karenge.
# Wo AI model words ka meaning samajhta hai.
#
# Example:
# blood sugar      -> glucose          ✅
# sex              -> gender           ✅
# bp               -> blood pressure   ✅
# body mass index  -> bmi              ✅
# disease          -> diagnosis        ✅
# smoking status   -> smoker           ✅
# Final Schema Mapping me dono scores combine honge:
#
# RapidFuzz (Spelling)
#          +
# Sentence Transformer (Meaning)
#          ↓
# Final Accurate Mapping
# ==========================================================

In [10]:
# ==========================================================
# STEP 4 : Sentence Transformer
# ==========================================================

# Sentence Transformer ek AI model hai jo text ka meaning samajhta hai.
# Ye text ko numbers (vectors/embeddings) me convert karta hai.
#
# RapidFuzz sirf spelling compare karta tha.
# Sentence Transformer meaning compare karega.
#
# Example:
#
# blood sugar  -> glucose           ✅ Same Meaning
# sex          -> gender            ✅ Same Meaning
# bp           -> blood pressure    ✅ Same Meaning
# bmi          -> body mass index   ✅ Same Meaning
#
# Isi wajah se Schema Mapper me semantic matching use ki jati hai.
# ==========================================================

# from sentence_transformers import SentenceTransformer

# SentenceTransformer library import ki.

# model = SentenceTransformer("all-MiniLM-L6-v2")

# "all-MiniLM-L6-v2" ek lightweight aur fast pretrained AI model hai.
# Ye text ko embeddings (vectors) me convert karega.

# print("Sentence Transformer Model Loaded Successfully!")

In [11]:
# ==========================================================
# NOTE:
#
# Embedding ka matlab:
#
# AI kisi bhi word ya sentence ko directly compare nahi karta.
#
# Pehle us text ko numbers (Vector) me convert karta hai.
#
# Ye numbers us text ka meaning represent karte hain.
#
# Example:
#
# blood sugar
# ↓
# [0.23, -0.51, ....]
#
# glucose
# ↓
# [0.19, -0.48, ....]
#
# Dono vectors agar similar honge to AI kahega
# ke dono words ka meaning bhi similar hai.
#
# Isi technique ko Semantic Similarity kehte hain.
#
# Agle step me hum Cosine Similarity use karenge.
# Wo in vectors ko compare karegi aur similarity score degi.
# ==========================================================

# Generate Embeddings for All Columns

In [12]:
# ==========================================================
# STEP 5 : Generate Embeddings
# ==========================================================

# NOTE:
# source_df aur target_df pehle hi standardize_columns()
# function se clean ho chuke hain.
# Ab Sentence Transformer in cleaned column names ko
# numerical vectors (embeddings) me convert karega.

# Get standardized column names
source_columns = source_df.columns.tolist()
target_columns = target_df.columns.tolist()

# Generate embeddings for source columns
source_embeddings = model.encode(source_columns)

# Generate embeddings for target columns
target_embeddings = model.encode(target_columns)

print("✅ Source Embeddings Created Successfully!")
print("✅ Target Embeddings Created Successfully!")

print(f"\nTotal Source Columns : {len(source_embeddings)}")
print(f"Total Target Columns : {len(target_embeddings)}")

✅ Source Embeddings Created Successfully!
✅ Target Embeddings Created Successfully!

Total Source Columns : 10
Total Target Columns : 10


In [13]:
# ==========================================================
# STEP 6 : Calculate Cosine Similarity
# ==========================================================

from sklearn.metrics.pairwise import cosine_similarity

# cosine_similarity() do vectors ke meaning compare karti hai.
# Ye 0 se 1 ke beech similarity score deti hai.


similarity_matrix = cosine_similarity(source_embeddings, target_embeddings)

# Source ke har embedding ko Target ke har embedding se compare karega.
# Result ek matrix hogi.


print("Cosine Similarity Matrix Created Successfully!")

# Matrix successfully create ho gayi.


print(similarity_matrix)

# Poori similarity matrix print karega.

Cosine Similarity Matrix Created Successfully!
[[ 1.          0.17321889  0.12627794  0.03408522  0.09748186  0.11090474
   0.08442897 -0.00249597  0.11840071  0.08649325]
 [ 0.14028758  0.5834521   0.22617634  0.06273405  0.25771827  0.11997841
   0.20236182  0.1175314   0.14607814  0.302549  ]
 [ 0.12627794  0.43850052  1.          0.1697262   0.16368416  0.26966488
   0.21256948  0.12599896  0.22963627  0.18793789]
 [ 0.03408522  0.10391983  0.1697262   1.0000001   0.22258411  0.23613249
   0.6714496   0.29001892  0.08562614  0.14100608]
 [ 0.13098823  0.12907794  0.15649836  0.15818009  0.6354241   0.33482367
   0.18529838  0.28382447  0.11448883  0.09746124]
 [ 0.17581446  0.0909263   0.2860973   0.18528628  0.23669633  0.54917336
   0.273805    0.13634159  0.03764687  0.11959963]
 [ 0.01848338  0.04906156  0.10879001  0.52167535  0.31129116  0.20395195
   0.8285173   0.2265149  -0.06634852  0.24460325]
 [-0.02765398  0.11779679  0.03129445  0.13257262  0.1247219   0.1829201
   0.

In [14]:
# ==========================================================
# Convert Similarity Matrix into DataFrame
# ==========================================================

import pandas as pd

# Similarity matrix ko readable table me convert karenge.


similarity_df = pd.DataFrame(
    similarity_matrix,
    index=source_columns,
    columns=target_columns
)

# Rows = Source Columns
# Columns = Target Columns


print(similarity_df)

# Similarity table print karega.

                   record id       age    gender   glucose  bloodpressure  \
record id           1.000000  0.173219  0.126278  0.034085       0.097482   
patient age         0.140288  0.583452  0.226176  0.062734       0.257718   
gender              0.126278  0.438501  1.000000  0.169726       0.163684   
glucose             0.034085  0.103920  0.169726  1.000000       0.222584   
blood pressure      0.130988  0.129078  0.156498  0.158180       0.635424   
body mass index     0.175814  0.090926  0.286097  0.185286       0.236696   
serum insulin       0.018483  0.049062  0.108790  0.521675       0.311291   
cholesterol level  -0.027654  0.117797  0.031294  0.132573       0.124722   
smoking status      0.124453  0.201410  0.086718  0.041078       0.105245   
disease             0.191365  0.279766  0.263599  0.269865       0.265039   

                        bmi   insulin  cholesterol    smoker  diagnosis  
record id          0.110905  0.084429    -0.002496  0.118401   0.086493  
pati

In [15]:
# ==========================================================
# STEP 7 : Generate Schema Mapping (Generic One-to-One Mapping)
# ==========================================================

print("========== SCHEMA MAPPING ==========\n")

# Final mapping store karne ke liye empty list banayi.
mapping = []

# Jo target columns already use ho chuke hain unko store karega.
used_targets = set()


# Har source column ke liye loop chalega.
for i in range(len(source_columns)):

    # Current source column
    source = source_columns[i]

    # Current source ki similarity scores nikali.
    scores = similarity_matrix[i]

    # Similarity scores ko highest se lowest order me sort kiya.
    sorted_index = scores.argsort()[::-1]


    # Best available target dhoondhne ke liye loop.
    for index in sorted_index:

        # Current target column
        target = target_columns[index]

        # Similarity score
        score = scores[index]

        # Agar target pehle use nahi hua to usi ko assign kar do.
        if target not in used_targets:

            # Similarity score ke basis par mapping status decide kiya.
            if score >= 0.70:
                status = "Matched"

            elif score >= 0.50:
                status = "Review Required"

            else:
                status = "Low Confidence"


            # Mapping list me record add kiya.
            mapping.append({

                "Source Column": source,

                "Target Column": target,

                "Similarity Score": round(float(score), 3),

                "Status": status

            })


            # Assigned target ko used list me add kar diya.
            used_targets.add(target)

            # Best target mil gaya, isliye loop stop.
            break


# Mapping list ko DataFrame me convert kiya.
mapping_df = pd.DataFrame(mapping)


print("Schema Mapping Completed Successfully!\n")

display(mapping_df)


# ==========================================================
# NOTE:
#
# Is step me har source column ke liye
# sabse suitable target column choose kiya gaya.
#
# Similarity scores ko highest se lowest order me sort kiya gaya.
#
# Agar koi target column pehle hi assign ho chuka ho,
# to usko dobara use nahi kiya gaya.
#
# Similarity Score ke basis par har mapping ko
# ek Status assign kiya gaya:
#
# Matched           → Score >= 0.70
#
# Review Required   → Score 0.50 se 0.69
#
# Low Confidence    → Score < 0.50
#
# Is approach ko One-to-One Mapping kehte hain.
#
# Ye approach kisi bhi do CSV schemas ke liye
# generic tarike se kaam kar sakti hai.
# ==========================================================

========== SCHEMA MAPPING ==========

Schema Mapping Completed Successfully!



,Source Column,Target Column,Similarity Score,Status
0,record id,record id,1.000,Matched
1,patient age,age,0.583,Review Required
2,gender,gender,1.000,Matched
3,glucose,glucose,1.000,Matched
4,blood pressure,bloodpressure,0.635,Review Required
5,body mass index,bmi,0.549,Review Required
6,serum insulin,insulin,0.829,Matched
7,cholesterol level,cholesterol,0.831,Matched
8,smoking status,smoker,0.598,Review Required
9,disease,diagnosis,0.633,Review Required


In [16]:
# ==========================================================
# NOTE:
#
# Ab mapping sirf print nahi ho rahi.
#
# Humne final mapping ko DataFrame me store kar liya hai.
#
# Is DataFrame ko future me:
#
# ✔ JSON file me save kar sakte hain.
# ✔ CSV me export kar sakte hain.
# ✔ Streamlit table me show kar sakte hain.
# ✔ LLM ko input ke taur par de sakte hain.
#
# Ye DataFrame project ka main output hai.
# ==========================================================

In [17]:
# ==========================================================
# STEP 8 : Data Quality Check
# ==========================================================

# ==========================================================
# Source CSV Information
# ==========================================================

print("========== SOURCE DATA QUALITY ==========\n")

# Total rows aur columns batayega.
print("Shape :", source_df.shape)

# Har column ka data type batayega.
print("\nData Types")
print(source_df.dtypes)

# Har column me missing values count karega.
print("\nMissing Values")
print(source_df.isnull().sum())

# Duplicate rows count karega.
print("\nDuplicate Rows")
print(source_df.duplicated().sum())


# ==========================================================
# Target CSV Information
# ==========================================================

print("\n\n========== TARGET DATA QUALITY ==========\n")

print("Shape :", target_df.shape)

print("\nData Types")
print(target_df.dtypes)

print("\nMissing Values")
print(target_df.isnull().sum())

print("\nDuplicate Rows")
print(target_df.duplicated().sum())

========== SOURCE DATA QUALITY ==========

Shape : (2000, 10)

Data Types
record id              int64
patient age            int64
gender                object
glucose                int64
blood pressure         int64
body mass index      float64
serum insulin          int64
cholesterol level      int64
smoking status        object
disease               object
dtype: object

Missing Values
record id            0
patient age          0
gender               0
glucose              0
blood pressure       0
body mass index      0
serum insulin        0
cholesterol level    0
smoking status       0
disease              0
dtype: int64

Duplicate Rows
0


========== TARGET DATA QUALITY ==========

Shape : (2000, 10)

Data Types
record id          int64
age                int64
gender            object
glucose            int64
bloodpressure      int64
bmi              float64
insulin            int64
cholesterol        int64
smoker            object
diagnosis         object
dtype: object

Miss

In [18]:
# ==========================================================
# STEP 9 : Data Cleaning Function
# ==========================================================

def clean_dataset(df):

    # -------------------------------
    # 1. Remove Duplicate Rows
    # -------------------------------

    df = df.drop_duplicates()

    # Duplicate rows remove kar di.


    # -------------------------------
    # 2. Handle Missing Values
    # -------------------------------

    # Numeric columns identify kiye.
    numeric_columns = df.select_dtypes(include="number").columns

    # Missing numeric values ko median se fill kiya.
    df[numeric_columns] = df[numeric_columns].fillna(
        df[numeric_columns].median()
    )


    # Text columns identify kiye.
    text_columns = df.select_dtypes(include="object").columns

    # Missing text values ko mode se fill kiya.
    df[text_columns] = df[text_columns].fillna(
        df[text_columns].mode().iloc[0]
    )


    # -------------------------------
    # 3. Remove Extra Spaces
    # -------------------------------

    for column in text_columns:

        df[column] = df[column].str.strip()

        # Starting aur ending spaces remove ki.


    # -------------------------------
    # 4. Convert Text to Lowercase
    # -------------------------------

    for column in text_columns:

        df[column] = df[column].str.lower()

        # Text ko lowercase me convert kiya.


    # -------------------------------
    # 5. Handle Outliers (IQR Method)
    # -------------------------------

    for column in numeric_columns:

        Q1 = df[column].quantile(0.25)

        # First Quartile

        Q3 = df[column].quantile(0.75)

        # Third Quartile

        IQR = Q3 - Q1

        # Inter Quartile Range


        lower_limit = Q1 - (1.5 * IQR)

        # Minimum acceptable value


        upper_limit = Q3 + (1.5 * IQR)

        # Maximum acceptable value


        df[column] = df[column].clip(lower_limit, upper_limit)

        # Outliers ko lower aur upper limit ke andar le aya.


    # -------------------------------
    # 6. Reset Index
    # -------------------------------

    df.reset_index(drop=True, inplace=True)

    # Index dobara 0 se start hoga.


    # Cleaned dataset return karega.
    return df

In [19]:
# ==========================================================
# Clean Source Dataset
# ==========================================================

source_df = clean_dataset(source_df)

# Source dataset clean kiya.


# ==========================================================
# Clean Target Dataset
# ==========================================================

target_df = clean_dataset(target_df)

# Target dataset clean kiya.


print("Source Dataset Cleaned Successfully!")

print("Target Dataset Cleaned Successfully!")

print("\nSource Shape :", source_df.shape)

print("Target Shape :", target_df.shape)

Source Dataset Cleaned Successfully!
Target Dataset Cleaned Successfully!

Source Shape : (2000, 10)
Target Shape : (2000, 10)


# Create Transformed Dataset

In [20]:
# ==========================================================
# STEP 11 : Data Transformation
# ==========================================================

print("========== DATA TRANSFORMATION ==========\n")

# Empty DataFrame banaya.
transformed_df = pd.DataFrame()

# Is DataFrame me target schema ke according data store hoga.


# Mapping DataFrame ki har row par loop chalega.
for i in range(len(mapping_df)):

    # Source column ka naam.
    source_column = mapping_df.loc[i, "Source Column"]

    # Target column ka naam.
    target_column = mapping_df.loc[i, "Target Column"]

    # Mapping ka status.
    status = mapping_df.loc[i, "Status"]


    # Sirf Matched aur Review Required mappings ko transform karenge.
    if status != "Low Confidence":

        # Agar source column dataset me exist karta hai.
        if source_column in source_df.columns:

            # Source column ka data target column ke naam se store kar diya.
            transformed_df[target_column] = source_df[source_column]


print("Data Transformation Completed Successfully!\n")

print("Transformed Dataset Shape :", transformed_df.shape)

print("\nTransformed Dataset Preview\n")

display(transformed_df.head())


# ==========================================================
# NOTE:
#
# Is step me source dataset ko
# target schema ke according convert kiya gaya.
#
# Mapping DataFrame ki information use ki gayi.
#
# Sirf Matched aur Review Required
# columns ko transform kiya gaya.
#
# Low Confidence mappings ko
# automatically skip kar diya gaya.
#
# Final transformed dataset
# target schema follow karta hai.
#
# ==========================================================

========== DATA TRANSFORMATION ==========

Data Transformation Completed Successfully!

Transformed Dataset Shape : (2000, 10)

Transformed Dataset Preview



,record id,age,gender,glucose,bloodpressure,bmi,insulin,cholesterol,smoker,diagnosis
0,10001,61,m,76,107,24.6,129,155,smoker,negative
1,10002,64,m,178,62,18.7,126,179,smoker,positive
2,10003,56,m,209,86,23.3,157,121,smoker,positive
3,10004,65,f,157,77,21.7,187,146,smoker,positive
4,10005,45,m,161,82,32.5,37,237,smoker,positive


# Verify Transformation

In [21]:
# ==========================================================
# STEP 11.1 : Verify Transformation
# ==========================================================

print("========== TRANSFORMATION VALIDATION ==========\n")

print("Target Columns")

print(list(target_df.columns))

print("\nTransformed Columns")

print(list(transformed_df.columns))


# Dono schemas compare kiye.
if list(target_df.columns) == list(transformed_df.columns):

    print("\nTransformation Successful")

else:

    print("\nTransformation Failed")

========== TRANSFORMATION VALIDATION ==========

Target Columns
['record id', 'age', 'gender', 'glucose', 'bloodpressure', 'bmi', 'insulin', 'cholesterol', 'smoker', 'diagnosis']

Transformed Columns
['record id', 'age', 'gender', 'glucose', 'bloodpressure', 'bmi', 'insulin', 'cholesterol', 'smoker', 'diagnosis']

Transformation Successful


# Save Transformed Dataset

In [22]:
# ==========================================================
# STEP 12 : Save Transformed Dataset
# ==========================================================

# Transformed dataset ko CSV file me save kiya.
transformed_df.to_csv("transformed_dataset.csv", index=False)

# index=False ka matlab row numbers save nahi honge.


print("Transformed Dataset Saved Successfully!")

print("File Name : transformed_dataset.csv")


# ==========================================================
# NOTE:
#
# to_csv() DataFrame ko CSV file me save karta hai.
#
# "transformed_dataset.csv"
# output file ka naam hai.
#
# index=False
# row numbers ko CSV me save nahi karta.
#
# Ye final transformed dataset hai jo
# target schema ke according generate hua hai.
# ==========================================================

Transformed Dataset Saved Successfully!
File Name : transformed_dataset.csv


In [23]:
import json

# ==========================================================
# STEP 13 : Generate JSON Mapping Report
# ==========================================================

print("========== GENERATING JSON REPORT ==========\n")

# Mapping summary calculate ki.
matched = (mapping_df["Status"] == "Matched").sum()

review = (mapping_df["Status"] == "Review Required").sum()

low = (mapping_df["Status"] == "Low Confidence").sum()


# Final JSON report banaya.
mapping_report = {

    "Total Source Columns": len(source_columns),

    "Total Target Columns": len(target_columns),

    "Total Mappings": len(mapping_df),

    "Matched": int(matched),

    "Review Required": int(review),

    "Low Confidence": int(low),

    "Average Similarity Score": round(
        mapping_df["Similarity Score"].mean(), 3
    ),

    "Mapping Details": mapping_df.to_dict(orient="records")

}


# JSON file save ki.
with open("schema_mapping_report.json", "w", encoding="utf-8") as file:

    json.dump(
        mapping_report,
        file,
        indent=4,
        ensure_ascii=False
    )


print("JSON Report Generated Successfully!")

print("File Name : schema_mapping_report.json")


# ==========================================================
# NOTE:
#
# Is step me mapping_df ki information
# JSON format me save ki gayi hai.
#
# Mapping status dobara calculate nahi kiya gaya.
#
# Status seedha mapping_df se liya gaya hai.
#
# Average Similarity Score bhi report me add kiya gaya hai.
#
# ensure_ascii=False ki wajah se future me
# Unicode characters bhi sahi save honge.
#
# ==========================================================

========== GENERATING JSON REPORT ==========

JSON Report Generated Successfully!
File Name : schema_mapping_report.json
